In [1]:
import pandas as pd

In [2]:
# load in all the cleaned dataframes from the clean folder
results = pd.read_csv('../data/clean/results.csv')
races = pd.read_csv('../data/clean/races.csv')
pit_stops = pd.read_csv('../data/clean/pit_stops.csv')
drivers = pd.read_csv('../data/clean/drivers.csv')  
constructors = pd.read_csv('../data/clean/constructors.csv')

Setup: recast dtypes on load. CSV format does not preserve nullable Int64 metadata, so position, number, and duration are re-cast using the same dtypes_dict pattern from the cleaning notebook

In [3]:
# create a dictionary of the dataframes
dataframes = {
    'results': results,
    'races': races,
    'pit_stops': pit_stops,
    'drivers': drivers,
    'constructors': constructors
}

In [4]:
dtypes_dict = {
    'results': {
        'position': 'Int64'
    },
    'drivers': {
        'number': 'Int64'
    },
    'pit_stops': {
        'duration': 'float64'
    }
}

In [5]:
for name, cols in dtypes_dict.items():
    dataframes[name] = dataframes[name].astype(cols)

In [6]:
# Reassign the recast DataFrames back to the standalone variables
results = dataframes['results']
drivers = dataframes['drivers']
pit_stops = dataframes['pit_stops']

## Feature 1: position_change

**Formula:** `position_change = grid - position`

**Type:** Driver skill proxy (primary)

### What It Measures
The number of positions a driver gained or lost during the race. Positive means the driver gained places (started 10th, finished 6th gives +4). Negative means the driver lost places. Zero means they finished where they started.

### Why It Is a Driver Skill Proxy
The starting grid already prices in the car. Qualifying sorts the field by car pace, so by the time the lights go out, car speed is baked into the grid slot. Any movement after that points to what the driver did on Sunday: overtakes, defending, tire management, racecraft.

### Null Decision 1: DNFs Stay Null
A driver who did not finish has no finishing position, so positions gained is undefined for them. `grid - position` propagates the null automatically (any math with pd.NA returns pd.NA), so no special handling was needed. Assigning a 0 instead would claim "crashed on lap 2" and "held position all race" are the same performance, and would violate the no-imputation principle from the cleaning notebook.

### Null Decision 2: Pit Lane Starts (grid = 0) Set to Null
In this dataset, grid = 0 is a code for a pit lane start, not a real grid slot. Left alone, the formula produces upside-down results: a driver who started from the pit lane (behind everyone) and climbed to P15 would show -15, claiming they collapsed when they actually gained places. 85 rows (about 1.3% of results) were affected.

Fix applied with a vectorized boolean mask, no loops:

`results.loc[results['grid'] == 0, 'position_change'] = pd.NA`

Justification: grid = 0 is a code, not a position, so position_change is undefined for pit lane starts. Inventing a starting position (for example treating them as P20) would be imputation.

### Verification: Three Checks, Three Claims
Each null decision gets its own proof, and when two mechanisms can overlap on the same rows, a check is added to separate them.

1. DNF check (position is null): proves null propagation worked for drivers who started normally but did not finish. These rows have real grid numbers, so grid = 0 is not involved.
2. All grid = 0 rows: proves the .loc fix ran. On its own this check has a blind spot, because pit lane starters who also DNFed were already null from propagation before the fix ran.
3. Grid = 0 AND position is not null: closes the blind spot. These are pit lane starters who finished the race, so propagation could not have made them null. The only way these rows show NA is if the .loc fix did its job. This check isolates the fix as the cause.

### What the Nulls Mean Downstream
The row stays, one cell sits out. A pit lane starter or DNF row still contributes its points to points_per_race and teammate_gap, its statusId to the DNF story, and its pit stops to avg_pit_stop_per_team_per_season. Only position_change is benched for that row. Pandas aggregations (.mean(), .corr(), groupby) skip nulls by default, so era comparisons and correlations are computed only on honest values, roughly 5,300+ usable rows out of 6,436.

### Limitation
position_change only measures finished races from a numbered grid slot. A driver who crashes frequently looks fine on this metric because their DNFs vanish from it. This is covered by pairing it with teammate_gap, which is built on points, where a DNF scores zero for real. Team pit strategy calls can also gain or lose positions, so this proxy is not purely driver skill. Documented tradeoff, used alongside other proxies.

In [7]:
# create position_change
results['position_change'] = results['grid'] - results['position']

In [8]:
(results['grid'] == 0).sum()

85

In [9]:
results.loc[results['grid'] == 0, 'position_change'] = pd.NA

In [10]:
results[results['grid'] == 0][['grid', 'position', 'position_change']].head()

,grid,position,position_change
931,0,<NA>,<NA>
932,0,<NA>,<NA>
1100,0,<NA>,<NA>
2231,0,<NA>,<NA>
2251,0,<NA>,<NA>


In [12]:
# Verify null propagation for DNF rows
results[results['position'].isna()][['grid', 'position', 'position_change']].head()

,grid,position,position_change
17,14,<NA>,<NA>
18,23,<NA>,<NA>
19,19,<NA>,<NA>
20,17,<NA>,<NA>
21,16,<NA>,<NA>


In [11]:
results[(results['grid'] == 0) & (results['position'].notna())][['grid', 'position', 'position_change']].head()

,grid,position,position_change
2321,0,10,<NA>
2787,0,20,<NA>
3926,0,10,<NA>
3946,0,10,<NA>
3952,0,16,<NA>
